## 이벤트별 집계 확인

이 코드는 사용자의 서비스 전환 과정을 파악하기 위해, 퍼널(Funnel) 분석 대상이 될 핵심 이벤트를 선별하고 집계하는 단계임. **AARRR 프레임워크의 Activation(활성화) 단계**에서 사용자가 핵심 가치를 경험하는 지점을 파악하는 데 활용됨.

* **`event_name` 기준 집계:** 서비스 내 발생하는 전체 이벤트 종류 및 발생 건수 확인
* **사용자 수(`unique_users`) 함께 확인:** 단순히 총 발생 수만 보는 것이 아니라, 실제 해당 행동을 경험한 사용자 기반의 규모 측정

### 주요 확인 포인트
* **이벤트 라인업 파악:** Activation 퍼널을 구성할 핵심 행동(이벤트)이 정상적으로 수집되고 있는지 확인
* **Activation 퍼널 이벤트 선별:** 가입 이후 첫 핵심 기능 이용, 주요 페이지 방문 등 퍼널 단계별 기준이 될 이벤트를 결정


In [1]:
from google.cloud import bigquery

client = bigquery.Client(project="pro-talon-503713-s3")

query = """
SELECT
    event_name,
    COUNT(*) AS event_count,
    COUNT(DISTINCT user_pseudo_id) AS unique_users
FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
WHERE _TABLE_SUFFIX BETWEEN '20201101' AND '20210131'
GROUP BY event_name
ORDER BY event_count DESC
"""

df = client.query(query).to_dataframe()
print(df)

             event_name  event_count  unique_users
0             page_view      1350428        269792
1       user_engagement      1058721        213004
2                scroll       493072        138098
3             view_item       386068         61252
4         session_start       354970        267116
5           first_visit       257462        257314
6        view_promotion       190104        102443
7           add_to_cart        58543         12545
8        begin_checkout        38757          9715
9           select_item        31007         13180
10  view_search_results        26172         14449
11    add_shipping_info        19722          9714
12     add_payment_info        13899          5751
13     select_promotion         9450          8164
14             purchase         5692          4419
15                click         1446          1010
16       view_item_list           71            44


# GA4 주요 이벤트 정의
Google Analytics 4(GA4)에서 수집되는 주요 이벤트를 공식 이벤트 정의 기준으로 정리한 문서입니다.

| 구분 | 이벤트 | 정의 및 발생 조건 | 분석 시 주요 의미 |
|:---:|:---|:---|:---|
| 자동 수집 | `page_view` | 페이지가 로드되거나, 활성 사이트에서 브라우저 기록 상태가 변경될 때 발생합니다. | 페이지 조회 및 화면 전환량 측정 |
| 자동 수집 | `scroll` | 사용자가 각 페이지에서 처음으로 하단에 도달했을 때 발생합니다. 웹에서는 세로 기준으로 페이지의 90% 이상이 표시된 경우를 의미합니다. | 콘텐츠 도달 및 페이지 참여도 측정 |
| 자동 수집 | `session_start` | 사용자가 앱 또는 웹사이트에 참여할 때 발생합니다. 세션 ID와 세션 번호는 세션마다 자동으로 생성되며 세션의 각 이벤트에 연결됩니다. | 세션 수와 방문 시작 시점 파악 |
| 자동 수집 | `user_engagement` | 앱이 포그라운드에 있거나 웹페이지가 최소 1초간 포커스 상태일 때 발생합니다. | 실제 사용자 참여 시간 및 활성도 측정 |
| 자동 수집 | `view_search_results` | URL에 검색어 쿼리 매개변수가 포함되어 사용자가 사이트 검색을 수행한 것으로 간주될 때마다 발생합니다. | 사이트 검색 이용 현황 및 검색어 분석 |
| 자동 수집 | `first_visit` | 사용자가 웹사이트를 처음 방문하거나, 애널리틱스를 사용하는 Android 인스턴트 앱을 처음 실행할 때 발생합니다. | 신규 사용자 유입 및 최초 방문 분석 |
| 전자상거래 | `add_payment_info` | 사용자가 결제 과정에서 결제 정보를 제출할 때 발생합니다. | 결제 정보 입력 단계의 진행 및 이탈 분석 |
| 전자상거래 | `add_shipping_info` | 사용자가 결제 과정에서 배송 정보를 제출할 때 발생합니다. | 배송 정보 입력 단계의 진행 및 이탈 분석 |
| 전자상거래 | `add_to_cart` | 사용자가 상품을 장바구니에 추가할 때 발생합니다. | 상품 관심도 및 장바구니 전환 분석 |
| 전자상거래 | `begin_checkout` | 사용자가 결제 프로세스를 시작할 때 발생합니다. | 장바구니에서 결제로 이동한 사용자 분석 |
| 전자상거래 | `purchase` | 사용자가 구매를 완료할 때 발생합니다. | 매출, 구매 건수 및 구매 전환 분석 |
| 전자상거래 | `view_item` | 사용자가 상품을 조회할 때 발생합니다. | 개별 상품 조회 및 상품 관심도 분석 |
| 전자상거래 | `view_item_list` | 사용자가 상품 또는 서비스 목록을 조회할 때 발생합니다. | 목록 노출 및 상품 탐색 분석 |
| 프로모션 | `view_promotion` | 사용자가 웹사이트 또는 앱에서 프로모션을 조회할 때 발생합니다. | 프로모션 노출 및 도달 분석 |
| 프로모션 | `select_promotion` | 사용자가 프로모션을 선택할 때 발생합니다. | 프로모션 클릭 및 프로모션 유입 분석 |
| 상품 목록 상호작용 | `select_item` | 사용자가 상품 또는 서비스 목록에서 항목을 선택할 때 발생합니다. | 목록에서 개별 상품으로 이동한 행동 분석 |
| 사용자 상호작용 | `click` | 사용자가 현재 도메인에서 나가는 링크를 클릭할 때마다 발생합니다. | 외부 링크 클릭 및 이탈 경로 분석 |